# AI-Powered Crop Recommendation System with Intelligent Advisory

## Problem Statement

Farmers often struggle to select the most suitable crop due to:
- Lack of understanding of soil nutrient composition
- Changing environmental conditions
- Absence of data-driven decision tools

This leads to:
- Poor yield
- Soil degradation
- Financial losses

## Solution

We build an **AI-powered decision system** that:

 * Recommends the best crop based on soil nutrients  
* Explains WHY the crop is recommended (Explainable AI)  
* Provides actionable farming suggestions  
* Simulates an intelligent AI agent for interaction  



## Key Features

- Machine Learning Classification Models
- Feature Engineering (Soil Fertility & Ratios)
- Explainable AI (Feature Importance)
- AI Agent Simulation (Interactive Advisory)



## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

## Load Dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Agri_DS/sensor_Crop_Dataset (1).csv")  # adjust filename if needed

print("Shape:", df.shape)
df.head()

## Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)

print("\nData Info:")
df.info()

print("\nUnique Crops:\n", df['Crop'].unique())
print("\nSoil Types:\n", df['Soil_Type'].unique())
print("\nVarieties:\n", df['Variety'].unique())

In [ ]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)

print("\nData Info:")
df.info()

print("\nUnique Crops:\n", df['Crop'].unique())
print("\nSoil Types:\n", df['Soil_Type'].unique())
print("\nVarieties:\n", df['Variety'].unique())

## Dataset Overview

This dataset includes:

### Numerical Features
- Nitrogen, Phosphorus, Potassium
- Temperature, Humidity, pH_Value, Rainfall

### Categorical Features
- Soil_Type → Type of soil
- Variety → Crop variety

### Target Variable
- Crop → Crop type (multi-class classification)

* This is a **real-world mixed-data ML problem**

## Basic EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))
sns.countplot(x='Crop', data=df)
plt.xticks(rotation=90)
plt.title("Crop Distribution")
plt.show()

## Correlation

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.select_dtypes(include=['float64']).corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

## Key Insights

- Nutrient levels (N, P, K) strongly influence crop selection
- Rainfall and humidity impact crop suitability
- Soil type adds contextual decision-making power

*  We enhance the dataset using feature engineering

## Feature Engineering

In [ ]:
df['fertility_score'] = df['Nitrogen'] + df['Phosphorus'] + df['Potassium']

df['np_ratio'] = df['Nitrogen'] / (df['Phosphorus'] + 1)
df['nk_ratio'] = df['Nitrogen'] / (df['Potassium'] + 1)
df['pk_ratio'] = df['Phosphorus'] / (df['Potassium'] + 1)

df.head()

## ENCODING CATEGORICAL FEATURES

In [ ]:
df_encoded = pd.get_dummies(df, columns=['Soil_Type', 'Variety'])

## SPLIT FEATURES & TARGET

In [ ]:
X = df_encoded.drop('Crop', axis=1)
y = df_encoded['Crop']

print("Feature Shape:", X.shape)

## SCALING

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## TRAIN TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## MODEL TRAINING

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(f"\n{name}")
    print("Accuracy:", acc)
    print(classification_report(y_test, preds))

## BEST MODEL

In [ ]:
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("Best Model:", best_model_name)

## CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix

preds = best_model.predict(X_test)

plt.figure(figsize=(10,6))
sns.heatmap(confusion_matrix(y_test, preds), cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

## FEATURE IMPORTANCE (Explainable AI)

In [ ]:
import pandas as pd

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_

    feature_importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print(feature_importance_df.head(10))

    plt.figure(figsize=(10,6))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
    plt.title("Top Important Features")
    plt.show()

## AI AGENT (CORE)

In [ ]:
def crop_advisor_agent(N, P, K, soil_type, variety,
                      temp=25, humidity=60, ph=6.5, rainfall=100):

    fertility = N + P + K
    np_ratio = N / (P + 1)
    nk_ratio = N / (K + 1)
    pk_ratio = P / (K + 1)

    input_dict = {
        'Nitrogen': N,
        'Phosphorus': P,
        'Potassium': K,
        'Temperature': temp,
        'Humidity': humidity,
        'pH_Value': ph,
        'Rainfall': rainfall,
        'fertility_score': fertility,
        'np_ratio': np_ratio,
        'nk_ratio': nk_ratio,
        'pk_ratio': pk_ratio
    }

    # Fill missing columns
    for col in X.columns:
        if col not in input_dict:
            input_dict[col] = 0

    # Encode categorical
    soil_col = f"Soil_Type_{soil_type}"
    variety_col = f"Variety_{variety}"

    if soil_col in input_dict:
        input_dict[soil_col] = 1

    if variety_col in input_dict:
        input_dict[variety_col] = 1

    input_df = pd.DataFrame([input_dict])
    input_scaled = scaler.transform(input_df)

    prediction = best_model.predict(input_scaled)[0]

    # Explanation
    explanation = []
    if N > 80:
        explanation.append("High nitrogen supports fast plant growth.")
    if P > 60:
        explanation.append("Phosphorus helps root development.")
    if K > 60:
        explanation.append("Potassium improves crop resistance.")

    # Recommendations
    recommendations = []
    if N < 50:
        recommendations.append("Increase nitrogen using organic manure.")
    if P < 40:
        recommendations.append("Add phosphorus fertilizers.")
    if K < 40:
        recommendations.append("Improve potassium using potash.")

    return prediction, explanation, recommendations

## TEST AGENT

In [ ]:
crop, explanation, recommendations = crop_advisor_agent(
    90, 40, 40, soil_type="Loamy", variety="Default"
)

print("🌾 Recommended Crop:", crop)

print("\n🧠 Explanation:")
for e in explanation:
    print("-", e)

print("\n📌 Recommendations:")
for r in recommendations:
    print("-", r)

## Gradio UI

In [ ]:
import gradio as gr

def gradio_agent(N, P, K, soil_type):
    crop, explanation, recommendations = crop_advisor_agent(
        N, P, K, soil_type, variety="Default"
    )

    return crop, "\n".join(explanation), "\n".join(recommendations)

interface = gr.Interface(
    fn=gradio_agent,
    inputs=[
        gr.Slider(0, 150, label="Nitrogen"),
        gr.Slider(0, 150, label="Phosphorus"),
        gr.Slider(0, 150, label="Potassium"),
        gr.Dropdown(list(df['Soil_Type'].unique()), label="Soil Type")
    ],
    outputs=["text", "text", "text"],
    title="🌾 AI Crop Advisor Agent"
)

interface.launch()

## Final System Capabilities

- Predicts best crop using ML
- Explains reasoning (Explainable AI)
- Provides actionable recommendations
- Simulates AI agent interaction



## Industry Impact

This can be deployed as:
- Farmer advisory app
- Government agri-support system
- Precision agriculture tool



## Why This Stands Out

- Real-world dataset (mixed features)
- Decision system, not just prediction
- Explainable + actionable AI
- AI Agent simulation

* This is a production-style AI system